# KDD: Online Retail Market-Basket Discovery

This notebook applies the full **Knowledge Discovery in Databases (KDD)** process to the uploaded Kaggle/UCI Online Retail dataset.

**Objective:** discover products frequently purchased together and recommend possible product bundles or cross-selling experiments.

KDD stages demonstrated:

1. Domain understanding and goal definition
2. Data selection
3. Data preprocessing
4. Data transformation
5. Data-mining task and algorithm selection
6. Pattern mining
7. Pattern evaluation and interpretation
8. Knowledge consolidation

In [ ]:
%pip -q install pandas numpy matplotlib seaborn mlxtend

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from mlxtend.frequent_patterns import apriori, association_rules
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")

## 1. KDD — Domain understanding and data selection

The retail business wants to identify products that are repeatedly purchased together. The result will be a shortlist of bundle and cross-selling hypotheses. It is not a causal model: a high-lift rule must be validated with margin, inventory, seasonality, and an experiment.

The official project source is the Kaggle E-Commerce Data page. The uploaded file contains invoice number, product code, description, quantity, date, unit price, customer ID, and country.

In [ ]:
filename="data.csv"
candidates=[Path(filename),Path("/content")/filename,Path("upload")/filename,Path("/workspace/scratch/16abedd5f8ac/upload")/filename,Path("projects/kdd/artifacts/data.csv")]
data_path=next((p for p in candidates if p.exists()),None)
if data_path is None:
    try:
        from google.colab import files
        uploaded=files.upload(); data_path=Path(next(iter(uploaded)))
    except Exception as exc:
        raise FileNotFoundError("Upload data.csv and rerun this cell") from exc
df=pd.read_csv(data_path,encoding="latin1")
print("Using:",data_path,"Shape:",df.shape)
display(df.head())

In [ ]:
profile=pd.DataFrame({"dtype":df.dtypes.astype(str),"missing":df.isna().sum(),"unique":df.nunique(dropna=False)})
display(profile)
print("Duplicate rows:",df.duplicated().sum())
print("Invoice IDs:",df["InvoiceNo"].nunique(),"Products:",df["Description"].nunique())
print("Countries:",df["Country"].nunique())

## 2. KDD — Data preprocessing

Preprocessing policy:

- Treat invoice numbers beginning with `C` as cancellations and exclude them from the positive-purchase basket.
- Remove missing product descriptions.
- Keep only positive quantities and positive unit prices.
- Remove exact duplicate rows.
- Use the United Kingdom as the initial market scope so support represents a coherent retail population.
- Keep `CustomerID` for optional customer-level analysis, but do not need it for invoice-level association mining.

In [ ]:
clean=df.copy()
clean["InvoiceNo"]=clean["InvoiceNo"].astype(str)
clean["Description"]=clean["Description"].astype("string").str.strip()
clean["InvoiceDate"]=pd.to_datetime(clean["InvoiceDate"],errors="coerce")
initial_rows=len(clean)
clean=clean.drop_duplicates()
clean=clean[clean["Description"].notna() & (clean["Description"]!="")]
clean=clean[~clean["InvoiceNo"].str.upper().str.startswith("C")]
clean=clean[(clean["Quantity"]>0)&(clean["UnitPrice"]>0)]
uk=clean[clean["Country"].eq("United Kingdom")].copy()
print("Initial rows:",initial_rows)
print("Cleaned rows:",len(clean))
print("UK rows:",len(uk))
print("UK transactions:",uk["InvoiceNo"].nunique())
print("UK products:",uk["Description"].nunique())

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,4))
top_products=uk["Description"].value_counts().head(15).sort_values()
top_products.plot(kind="barh",ax=axes[0],color="steelblue"); axes[0].set_title("Top products by line count")
uk.groupby("InvoiceDate")["InvoiceNo"].nunique().resample("ME").sum().plot(ax=axes[1],color="darkorange"); axes[1].set_title("Monthly transaction volume"); axes[1].set_ylabel("Transactions")
plt.tight_layout(); plt.show()

## 3. KDD — Data transformation

Association mining requires one row per transaction and one Boolean column per product. The basket value is `True` when a product appears in an invoice. Quantity is converted to presence/absence because the selected task is product co-occurrence, not quantity forecasting.

In [ ]:
basket=uk.groupby(["InvoiceNo","Description"])["Quantity"].sum().unstack(fill_value=0).gt(0)
print("Basket shape:",basket.shape)
display(basket.iloc[:5,:10].astype(int))
item_support=basket.mean().sort_values(ascending=False).head(15)
display(item_support.rename("support").to_frame().round(4))

## 4. KDD — Task and algorithm selection

The selected task is frequent-itemset and association-rule mining. Apriori is used because it is interpretable and directly exposes support, confidence, and lift.

Parameters for this demonstration:

- Minimum support: **0.03**
- Minimum confidence: **0.35**
- Ranking metric: **lift**

Support is the proportion of transactions containing an itemset. Confidence is the conditional frequency of the consequent given the antecedent. Lift compares that conditional frequency with the consequent baseline frequency.

In [ ]:
MIN_SUPPORT=.03; MIN_CONFIDENCE=.35
frequent=apriori(basket,min_support=MIN_SUPPORT,use_colnames=True,low_memory=True)
print("Frequent itemsets:",len(frequent))
display(frequent.sort_values(["support","itemsets"],ascending=[False,True]).head(15))
rules=association_rules(frequent,metric="confidence",min_threshold=MIN_CONFIDENCE)
if len(rules):
    rules=rules.sort_values(["lift","confidence","support"],ascending=False).reset_index(drop=True)
    rules["antecedents_text"]=rules["antecedents"].apply(lambda s:" | ".join(sorted(s)))
    rules["consequents_text"]=rules["consequents"].apply(lambda s:" | ".join(sorted(s)))
print("Association rules:",len(rules))
display(rules[["antecedents_text","consequents_text","support","confidence","lift"]].head(20).round(4))

## 5. KDD — Pattern evaluation and interpretation

Rules are ranked by lift, but they must also have enough support and confidence to be operationally useful. A high-lift rule can still be based on a small number of transactions. The next cells inspect the strongest rules and visualize support, confidence, and lift.

In [ ]:
if len(rules):
    plot_rules=rules.head(30).copy()
    plt.figure(figsize=(10,6)); sns.scatterplot(data=plot_rules,x="support",y="confidence",size="lift",hue="lift",palette="viridis",sizes=(50,400))
    plt.title("Top association rules: support, confidence, and lift"); plt.legend(bbox_to_anchor=(1.02,1),loc="upper left"); plt.tight_layout(); plt.show()
    print("Rule interpretation example:")
    row=rules.iloc[0]
    print("When customers buy [{}], the consequent [{}] appears with confidence {:.1%}; lift is {:.2f}; support is {:.1%}.".format(row["antecedents_text"],row["consequents_text"],row["confidence"],row["lift"],row["support"]))
else:
    print("No rules met the selected thresholds. Lower thresholds only with a documented business reason.")

## 6. KDD — Knowledge consolidation

The final artifact is a ranked recommendation table. Each row is a candidate for a bundle, recommendation placement, or cross-sell experiment. Before deployment, a merchandising team should verify margin, inventory, seasonality, product compatibility, and incremental impact through an A/B test. Association does not prove that showing a bundle will cause additional purchases.

In [ ]:
if len(rules):
    recommendations=rules[["antecedents_text","consequents_text","support","confidence","lift"]].head(30).copy()
    recommendations["recommendation"]="Test a bundle or cross-sell placement after checking margin, inventory, and product compatibility."
    display(recommendations.head(10).round(4))
else:
    recommendations=pd.DataFrame(columns=["antecedents_text","consequents_text","support","confidence","lift","recommendation"])

In [ ]:
artifact_dir=Path("kdd_artifacts"); artifact_dir.mkdir(exist_ok=True)
summary={"dataset":str(data_path),"initial_rows":int(initial_rows),"clean_rows":int(len(clean)),"market_scope":"United Kingdom","transactions":int(len(basket)),"unique_products":int(basket.shape[1]),"minimum_support":MIN_SUPPORT,"minimum_confidence":MIN_CONFIDENCE,"frequent_itemsets":int(len(frequent)),"rules":int(len(rules)),"knowledge_action":"Use ranked rules as merchandising hypotheses and validate them with margin, inventory, seasonality, and controlled experiments."}
(artifact_dir/"metrics.json").write_text(json.dumps(summary,indent=2))
frequent_out=frequent.copy(); frequent_out["itemsets_text"]=frequent_out["itemsets"].apply(lambda s:" | ".join(sorted(s))); frequent_out.drop(columns=["itemsets"]).to_csv(artifact_dir/"frequent_itemsets.csv",index=False)
recommendations.to_csv(artifact_dir/"recommendations.csv",index=False)
print("Saved:",[p.name for p in artifact_dir.iterdir()])

## Final KDD conclusion and verification checklist

This project selected a retail domain and target population, cleaned the transaction data, transformed it into a basket matrix, mined frequent itemsets and association rules, evaluated support/confidence/lift, and consolidated the results into merchandising recommendations.

Before submission, verify:

- The Kaggle E-Commerce Data page is cited.
- The uploaded data is loaded with `latin1` encoding.
- Cancellations, missing descriptions, invalid quantities/prices, and duplicates are addressed.
- The market scope is clearly stated as United Kingdom.
- Support and confidence thresholds are documented.
- Rules are interpreted as associations, not causal effects.
- Recommendations and metrics are saved as artifacts.
- Screenshots show the notebook output and final rule table.